# Modelo de Classificação: Random Forest para Predição de Popularidade
Este notebook documenta o desenvolvimento de um algoritmo de Aprendizado de Máquina Supervisionado (Random Forest) com o objetivo de classificar o potencial de popularidade de obras literárias no Booklog.

## 1. Preparação dos Dados e Treinamento Inicial
Nesta etapa, importamos o conjunto de dados pré-processado (`books_pivot_mapped.parquet`), que já contempla a consolidação de gêneros estruturada em formato binário.

Implementamos as seguintes diretrizes:
1. **Engenharia de Frequência:** Mensuração da relevância do autor com base no seu histórico no catálogo.
2. **Definição de Classes Alvo:** Estruturação em 3 categorias de popularidade (Bestseller, Média Popularidade e Nicho) utilizando o volume de avaliações.
3. **Treinamento e Balanceamento:** Aplicação do parâmetro `class_weight='balanced'` nativo do Random Forest para penalizar a classe majoritária e lidar com o desbalanceamento do dataset, dispensando técnicas de *undersampling*.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.under_sampling import RandomUnderSampler

# 1. Carregando os dados
df_rf = pd.read_parquet('../data/processed/books_pivot_mapped.parquet')

# 2. Criando o Alvo (As 3 Classes de Popularidade)
def definir_tres_classes(ratings):
    if ratings >= 10000:
        return 1  # Bestseller
    elif ratings >= 1000:
        return 2  # Média Popularidade
    else:
        return 3  # Nicho

df_rf['popularity_class'] = df_rf['totalratings'].apply(definir_tres_classes)

# 3. Tratando a coluna 'author' (Frequência), relevancia dos autores
autor_frequencia = df_rf['author'].value_counts()
df_rf['author_frequency'] = df_rf['author'].map(autor_frequencia)

# 4. Separando os Preditores (X) do Alvo (y)
# Excluímos os textos puros e as colunas que dão o gabarito, notas, etc..
colunas_proibidas = ['title', 'author', 'rating', 'totalratings', 'popularity_class']
X = df_rf.drop(columns=colunas_proibidas)
y = df_rf['popularity_class']

# 5. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Iniciando a Random Forest...")

# 6. O Random Forest
# n_estimators = 100 significa que estamos criando 100 árvores de decisão
# class_weight = 'balanced', é o balanceamento nativo do Random forest
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# Treinando o modelo
rf_model.fit(X_train, y_train)

# 7. Testando o modelo
previsoes = rf_model.predict(X_test)

# 8. Exibindo os Resultados
print("=-" * 35)
print("     RESULTADOS DO RANDOM FOREST (COM BALANCEAMENTO NATIVO)     ")
print("=" * 70)
print(f"Acurácia Global: {accuracy_score(y_test, previsoes) * 100:.2f}%\n")
print(classification_report(y_test, previsoes, target_names=['Classe 1', 'Classe 2', 'Classe 3']))
print("-=" * 35)

Iniciando a Random Forest...
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
     RESULTADOS DO RANDOM FOREST (COM BALANCEAMENTO NATIVO)     
Acurácia Global: 72.22%

              precision    recall  f1-score   support

    Classe 1       0.22      0.24      0.23      1106
    Classe 2       0.41      0.44      0.42      4995
    Classe 3       0.85      0.83      0.84     18493

    accuracy                           0.72     24594
   macro avg       0.49      0.50      0.50     24594
weighted avg       0.73      0.72      0.73     24594

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=


## 2. A Arena Justa: Aplicação de Undersampling Físico
Para garantir uma comparação metodologicamente justa e científica com os outros modelos de classificação, aplicamos a mesma técnica de balanceamento físico aos dados de treino deste modelo.


**Diretrizes desta etapa:**
1. **Remoção da Penalização Matemática:** O parâmetro `class_weight='balanced'` foi retirado do Random Forest, pois o conjunto de treino já se encontra perfeitamente balanceado.
2. **Avaliação de Trade-off:** O objetivo é observar o impacto do sacrifício da Acurácia Global (devido à perda massiva de dados da Classe 3) em prol de um aumento no *Recall* da Classe 1.

In [7]:
# Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Usando o undersampling para balancear
print("Aplicando Undersampling no conjunto de treino...")
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

print("Distribuição das classes após o Undersampling:")
print(y_train_resampled.value_counts())
print("-" * 30)

print("Plantando a Random Forest (Arena Justa)...")

#  O Random Forest
# Removemos o class_weight='balanced' porque as classes já estão iguais
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Treinando com os dados reduzidos e balanceados
rf_model.fit(X_train_resampled, y_train_resampled)

#  Testando o modelo na "prova final" (que não sofreu undersampling)
previsoes = rf_model.predict(X_test)

# Exibindo os Resultados
print("=-" * 35)
print("     RESULTADOS DO RANDOM FOREST (COM UNDERSAMPLING FÍSICO)     ")
print("=" * 70)
print(f"Acurácia Global: {accuracy_score(y_test, previsoes) * 100:.2f}%\n")
print(classification_report(y_test, previsoes, target_names=['Classe 1', 'Classe 2', 'Classe 3']))
print("-=" * 35)

Aplicando Undersampling no conjunto de treino...
Distribuição das classes após o Undersampling:
popularity_class
1    2582
2    2582
3    2582
Name: count, dtype: int64
------------------------------
Plantando a Random Forest (Arena Justa)...
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
     RESULTADOS DO RANDOM FOREST (COM UNDERSAMPLING FÍSICO)     
Acurácia Global: 58.97%

              precision    recall  f1-score   support

    Classe 1       0.14      0.59      0.22      1106
    Classe 2       0.32      0.43      0.37      4995
    Classe 3       0.90      0.63      0.74     18493

    accuracy                           0.59     24594
   macro avg       0.45      0.55      0.44     24594
weighted avg       0.75      0.59      0.64     24594

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=


## 3. Equilíbrio Inteligente: Balanced Random Forest
Esta abordagem utiliza o Balanced Random Forest Classifier para lidar com problema de perda de informação de forma inteligente: em vez de reduzir o banco de dados antes do treino, o algoritmo realiza um subamostramento dinâmico e aleatório para **cada árvore individual** construída na floresta. 

**Diretrizes desta etapa:**
1. **Preservação do Catálogo:** Nenhuma informação da classe majoritária (Nicho) é descartada globalmente. Como cada árvore recebe uma amostra diferente, a floresta inteira consegue aprender com a base de dados completa.
2. **Parâmetro Estratégico:** A configuração `sampling_strategy='not minority'` garante que o modelo foque em sub amostrar apenas as classes maiores para se igualarem à classe de interesse (Bestsellers).

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("Plantando uma Balanced Random Forest (Equilíbrio Inteligente)...")

# 1. Instanciando o modelo específico para dados desbalanceados
brf_model = BalancedRandomForestClassifier(n_estimators=100, random_state=42, sampling_strategy='not minority')

# 2. Treinando o modelo (Ele faz o undersampling internamente para cada árvore)
brf_model.fit(X_train, y_train)

# 3. Testando na prova final
previsoes_brf = brf_model.predict(X_test)

# 4. Exibindo os Resultados
print("=-" * 35)
print("     RESULTADOS DO BALANCED RANDOM FOREST CLASSIFIER     ")
print("======================================================================")
print(f"Acurácia Global: {accuracy_score(y_test, previsoes_brf) * 100:.2f}%\n")
print(classification_report(y_test, previsoes_brf, target_names=['Classe 1', 'Classe 2', 'Classe 3']))
print("-=" * 35)

Plantando uma Balanced Random Forest (Equilíbrio Inteligente)...
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
     RESULTADOS DO BALANCED RANDOM FOREST CLASSIFIER     
Acurácia Global: 62.38%

              precision    recall  f1-score   support

    Classe 1       0.13      0.62      0.22      1106
    Classe 2       0.38      0.44      0.41      4995
    Classe 3       0.92      0.67      0.78     18493

    accuracy                           0.62     24594
   macro avg       0.48      0.58      0.47     24594
weighted avg       0.77      0.62      0.68     24594

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=


## 4. Otimização Científica da Floresta
Para mitigar o risco de sobreajuste (*overfitting*) e aprimorar a identificação das classes minoritárias, realizamos uma busca sistemática de hiperparâmetros utilizando `GridSearchCV` com Validação Cruzada (5 *folds*).

Os parâmetros otimizados incluem:
* `max_depth`: Limitação do crescimento das árvores para garantir a generalização.
* `n_estimators`: Ampliação do volume de árvores para diluir erros individuais.
* `min_samples_split`: Exigência de amostras mínimas para justificar ramificações.

In [ ]:
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedRandomForestClassifier

print("Iniciando a Otimização Científica da Balanced Forest...")

# 1. O "Cardápio" de configurações
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Configurando o Buscador
# sampling_strategy='not minority' foca o balanceamento inteligente nas classes maiores
brf_base = BalancedRandomForestClassifier(random_state=42, sampling_strategy='not minority')

grid_search_brf = GridSearchCV(
    estimator=brf_base, 
    param_grid=param_grid, 
    scoring='f1_macro', 
    cv=5,               
    verbose=1, 
    n_jobs=-1           
)

# 3. Treinando com os DADOS ORIGINAIS DE TREINO
grid_search_brf.fit(X_train, y_train)

# 4. Extraindo o modelo Campeão
print("\n==================================================")
print("          MELHOR BALANCED FLORESTA ENCONTRADA")
print("==================================================")
print(f"Melhores parâmetros: {grid_search_brf.best_params_}")
print(f"F1-Score Médio de Validação: {grid_search_brf.best_score_ * 100:.2f}%")
print("==================================================\n")

# 5. Testando o modelo campeão na teste final
melhor_brf = grid_search_brf.best_estimator_
previsoes_otimizadas = melhor_brf.predict(X_test)

print("Relatório de Classificação Final da Balanced RF Otimizada:")
print(classification_report(y_test, previsoes_otimizadas, target_names=['Classe 1', 'Classe 2', 'Classe 3']))

Iniciando a Otimização Científica da Balanced Forest...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

          MELHOR BALANCED FLORESTA ENCONTRADA
Melhores parâmetros: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
F1-Score Médio de Validação: 49.58%

Relatório de Classificação Final da Balanced RF Otimizada:
              precision    recall  f1-score   support

    Classe 1       0.18      0.62      0.28      1106
    Classe 2       0.35      0.47      0.40      4995
    Classe 3       0.92      0.70      0.80     18493

    accuracy                           0.65     24594
   macro avg       0.48      0.60      0.49     24594
weighted avg       0.77      0.65      0.69     24594



## 5. Persistência dos Componentes para Produção
Para integração com o back-end do aplicativo, persistimos o modelo treinado e os dicionários auxiliares. Estes artefatos permitirão que o sistema receba dados de um novo livro e devolva sua classificação preditiva em tempo real.

In [ ]:
import joblib
import os

# 1. Caminho para a pasta 'models'
caminho_modelos = '../../Machine Learning/models'
os.makedirs(caminho_modelos, exist_ok=True)

# 2. Exportaçao do modelo random forest treinado
joblib.dump(melhor_brf, os.path.join(caminho_modelos, "RF_popularidade.pkl"))

# 3. Exportando o "Dicionário" de Autores
# Quando um usuário cadastrar um livro novo no app, o backend precisará consultar esse arquivo para saber qual é a "nota de frequência" do autor.
joblib.dump(autor_frequencia, os.path.join(caminho_modelos, "autor_frequencia_RF.pkl"))

print("Artefatos do Random Forest exportados com sucesso para a produção")

Artefatos do Random Forest exportados com sucesso para a produção
